# Laboratory 1

**Before you start:** 
1. install mmseqs2 using https://github.com/soedinglab/mmseqs2
2. run `fetch_data.sh`

In [ ]:
import subprocess
import collections
import sys
import pandas as pd

## Helpers

In [ ]:
def run(cmd, log_file=None):
    out_target = (
        open(log_file, "a", encoding="utf-8") if log_file else subprocess.DEVNULL
    )
    err_target = out_target if log_file else sys.stderr

    r = subprocess.run(cmd, shell=True, stdout=out_target, stderr=err_target, text=True)

    if log_file:
        out_target.close()

    if r.returncode:
        raise RuntimeError(f"Command failed with exit code {r.returncode}")

In [ ]:
def cluster_stats(tsv):
    """*_cluster.tsv is: representative <TAB> member"""
    n = collections.Counter(l.split("\t")[0] for l in open(tsv))
    s = list(n.values())
    return dict(
        clusters=len(n),
        avg_size=round(sum(s) / len(n), 2),
        singletons=sum(1 for x in s if x == 1),
        largest=max(s),
    )

## Setup

In [ ]:
run("rm -rf work && mkdir -p work")

MMSEQS = str("/opt/homebrew/bin/mmseqs")

print(
    subprocess.run(
        f"{MMSEQS} version", shell=True, capture_output=True, text=True
    ).stdout.strip()
)

## A — Clustering

- **Bit score** — how good the alignment is relative to a scoring system (higher is better)
- **E-value** — how many alignments with a score at least this good would be expected by chance in the search database (lower is better)
- **Sequence identity** — exact residue agreement (usually used with coverage parameter)

### `easy-cluster`

`--min-seq-id` is how *similar* the aligned part must be; `-c` is how *much* of
each sequence must be aligned. They are independent, and `--cov-mode` decides
which sequence the coverage applies to (`0` = both).

In [ ]:
%%time
run(
    f"{MMSEQS} easy-cluster data/globins.fasta work/clu50 work/tmp "
    f"--min-seq-id 0.5 -c 0.8 --cov-mode 0",
    "work/clu50.log",
)

In [ ]:
cluster_stats("work/clu50_cluster.tsv")

Three output files: 
- `_cluster.tsv`: representative (cluster centroid) → member
- `_rep_seq.fasta`: one sequence per cluster (**non-redundant set**)
- `_all_seqs.fasta`: cluster centroids and member sequences listed below

### `easy-linclust`

`cluster` prefilters then aligns every surviving pair; `linclust` only compares
sequences sharing a k-mer representative, which is O(N) instead of O(N²).

In [ ]:
%%time
run(
    f"{MMSEQS} easy-linclust data/globins.fasta work/lin50 work/tmp "
    f"--min-seq-id 0.5 -c 0.8 --cov-mode 0",
    "work/lin50.log",
)

In [ ]:
pd.DataFrame(
    {
        "cluster": cluster_stats("work/clu50_cluster.tsv"),
        "linclust": cluster_stats("work/lin50_cluster.tsv"),
    }
)

## B — Search and MSA

A FASTA file is not searchable: `createdb` builds the index structures first. Then we run `search` and `result2msa`.

In [ ]:
run("rm -rf work/msa && mkdir -p work/msa")

In [ ]:
log_file = "work/msa/msa.log"

run(f"{MMSEQS} createdb data/query.fasta work/msa/queryDB", log_file)
run(f"{MMSEQS} createdb data/target_1000.fasta work/msa/targetDB", log_file)
run(
    f"{MMSEQS} search work/msa/queryDB work/msa/targetDB work/msa/resultDB work/msa/tmp -e 1e-3",
    log_file,
)
run(
    f"{MMSEQS} convertalis work/msa/queryDB work/msa/targetDB work/msa/resultDB work/msa/hits.m8 "
    f"--format-output 'query,target,fident,alnlen,evalue,bits'",
    log_file,
)

msa_hits = pd.read_csv(
    "work/msa/hits.m8",
    sep="\t",
    names=["query", "target", "identity", "alnlen", "evalue", "bits"],
)

In [ ]:
msa_hits

In [ ]:
msa_hits.plot(
    x="evalue",
    y="identity",
    style=".-",
    figsize=(8, 4),
    ylabel="identity",
    legend=False,
    logx=True,
)
msa_hits.plot(
    x="bits", y="identity", style=".-", figsize=(8, 4), ylabel="identity", legend=False
);

`--msa-format-mode`: `2` = aligned FASTA, `5` = **a3m** (what HH-suite and
AlphaFold consume). Modes `0`/`1` are *binary* cA3M databases that `unpackdb`
cannot read — a common trap.

In [ ]:
run(
    f"{MMSEQS} result2msa work/msa/queryDB work/msa/targetDB work/msa/resultDB work/msa/msaDB "
    f"--msa-format-mode 2",
    log_file,
)
run(f"{MMSEQS} unpackdb work/msa/msaDB work/msa --unpack-name-mode 0", log_file)

for line in open("work/msa/0").read().splitlines():
    print(line[:78])

The beta-type globins align end to end; the alpha-type ones (HBA, HBAZ, HBAT)
open a gap at the N-terminus. That gap is the duplication that split the two
lineages.

---
# Exercises

## 1 — How does the identity threshold change the clustering?

- use both `cluster` and `linclust`
- run the above for `min-seq-id` in `[0.1, 0.2, 0.30, 0.50, 0.70, 0.90, 0.95]`
- estimate at which `min-seq-id` the clustering methods give divergent results

## 2 — How does the coverage and coverage mode change the clustering?

- use `cluster` with `min-seq-id 0.5`
- run the above for `cov` in `[0.50, 0.80, 0.95]` and `cov-mode` in `[0, 1, 2, 3]`
- can you spot anything interesting?